In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização grafica inline no Jupyter notebook
%matplotlib inline

# P300 visual: de eventos reais a predições em participantes retidos

Carregue três gravações da tarefa visual oddball do OpenNeuro ``ds005863`` atraves do EEGDash, inspecione seus codigos de evento e decodifique alvos em um novo participante. Os sujeitos 054, 119 e 123 totalizam cerca de 69 MB de arquivos de sinal. Defina ``EEGDASH_CACHE_DIR`` para reutilizar os downloads. A computação em CPU e suficiente; instale ``eegprep[eeglabio]>=0.2.23,<0.3`` para as etapas do EEGPrep. O conjunto de dados contem EEG gravado real; todos os rotulos decorrem de seus marcadores de estimulo.

Uma tarefa oddball apresenta estimulos frequentes (padrao) e raros (alvo). Um potencial evocado (ERP) calcula a media temporal alinhada ao estimulo; um decodificador, em contrapartida, preve a condição de cada ensaio individual. Aqui voce preparara ambos a partir das mesmas gravações e testara se um classificador simples baseado em amplitude se generaliza para um participante ausente do treino.



## 1. Selecionar as gravações antes do download
A consulta define três gravações. Inspecione os metadados antes de acessar ``raw``, que efetua o download.



In [ ]:
# Importa modulos de sistema operacional e manipulação de caminhos
import os
from pathlib import Path

# Importa bibliotecas para graficos, MNE, computação matricial e tabelas
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
# Importa rotinas de remoção de referencia media e offset DC do Braindecode
from braindecode.preprocessing import RemoveCommonAverageReference, RemoveDCOffset
# Importa regressão logistica, exibição de matriz de confusao e acuracia balanceada do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
# Importa divisao Leave-One-Group-Out, pipeline e padronizador de escala
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa dataset do EEGDash e função para calculo de media do sinal
from eegdash import EEGDashDataset
from eegdash.features import signal_mean

# Define diretorio de cache persistente
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
subjects = ["054", "119", "123"]
# Instancia o dataset carregando a tarefa visualoddball dos 3 sujeitos
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="ds005863",
    subject=subjects,
    task="visualoddball",
    n_jobs=1,
)
# Valida carregamento dos 3 sujeitos
assert len(dataset.datasets) == len(subjects)
print(dataset.description[["subject", "task"]])

## 2. Mapear eventos gravados e preparar caracteristicas de ensaios
No codigo XY, X e a letra-alvo do bloco e Y e a letra apresentada (ambas codificadas de 1 a 5). dígitos coincidentes denotam alvos (target = 2) e divergentes denotam padrao (standard = 1). Subtrair 1 gera as classes binarias 0 e 1.

As épocas do MNE cobrem de -0.1 a 0.8 s em relação ao estimulo com correção de linha de base de -100 a 0 ms. Filtramos de 0.5 a 30 Hz. O EEGPrep remove offset DC e aplica referencia media comum. Reamostramos para 128 Hz. A função ``signal_mean`` calcula a amplitude media entre 300 e 450 ms por canal e ensaio.



In [ ]:
features, labels, groups = [], [], []
first_epochs = None
channel_names = None
# Itera pelas gravações de cada participante
for recording in dataset.datasets:
    raw = recording.raw.copy().load_data().pick("eeg")
    mapping = {}
    # Mapeia eventos identificando se o dígito do estimulo coincide com o alvo do bloco
    for name in set(raw.annotations.description):
        code = name.split("/")[-1].replace(" ", "")
        if len(code) == 3 and code[0] == "S" and set(code[1:]) <= set("12345"):
            mapping[name] = 2 if code[1] == code[2] else 1
    assert set(mapping.values()) == {1, 2}, "Missing target or standard markers"
    print(recording.description["subject"], mapping)

    # Pre-processamento: salva anotações, remove DC offset, aplica CAR e filtra de 0.5 a 30 Hz
    source_annotations = raw.annotations.copy()
    source_date = raw.info["meas_date"]
    source_grid = (raw.info["sfreq"], raw.n_times, raw.first_samp)
    RemoveDCOffset().apply(raw)
    RemoveCommonAverageReference().apply(raw)
    assert (raw.info["sfreq"], raw.n_times, raw.first_samp) == source_grid
    raw.set_meas_date(source_date)
    raw.set_annotations(source_annotations)
    raw.filter(0.5, 30.0)
    # Extrai eventos e constroi as épocas com baseline (-100ms a 0ms)
    events, _ = mne.events_from_annotations(raw, event_id=mapping)
    epochs = mne.Epochs(
        raw,
        events,
        event_id={"standard": 1, "target": 2},
        tmin=-0.1,
        tmax=0.8,
        baseline=(-0.1, 0),
        preload=True,
        reject_by_annotation=True,
    )
    epochs.resample(128)
    if channel_names is None:
        channel_names = epochs.ch_names
        first_epochs = epochs
    assert epochs.ch_names == channel_names
    assert "Pz" in epochs.ch_names, "This ERP example requires Pz"
    X = epochs.get_data()
    y = epochs.events[:, 2] - 1
    assert set(y) == {0, 1} and np.isfinite(X).all()

    # Extrai amplitude media da janela de 300 a 450 ms (P300) em microvolts
    interval = (epochs.times >= 0.3) & (epochs.times <= 0.45)
    features.append(signal_mean(X[:, :, interval]) * 1e6)
    labels.append(y)
    groups.extend([str(recording.description["subject"])] * len(y))

## 3. Avaliar um participante retido (*held-out*) por partição
O StandardScaler é ajustado estritamente no treino de cada partição. A regressão logistica com ponderação balanceada (``class_weight="balanced"``) atribui pesos maiores aos alvos minoritarios na função de perda. A acuracia balanceada avalia a media da sensibilidade das duas condições.



In [ ]:
# Concatena as caracteristicas e rotulos de todos os participantes
X = np.concatenate(features)
y = np.concatenate(labels)
groups = np.asarray(groups)
# Exibe tabela cruzada de contagem de ensaios por classe e sujeito
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["class"]))
predictions = np.full(len(y), -1)
counts = np.zeros(len(y), dtype=int)
rows = []
# Validação cruzada deixando um participante de fora (LOSO)
for train, test in LeaveOneGroupOut().split(X, y, groups):
    assert set(groups[train]).isdisjoint(groups[test])
    model = make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)
    )
    model.fit(X[train], y[train])
    predictions[test] = model.predict(X[test])
    counts[test] += 1
    rows.append(
        {
            "subject": groups[test][0],
            "balanced_accuracy": balanced_accuracy_score(y[test], predictions[test]),
        }
    )
assert np.all(counts == 1)
# Exibe tabela com acuracia balanceada por participante retido
print(pd.DataFrame(rows).to_string(index=False))

## 4. Inspecionar o ERP medido e os erros de decodificação
Visualizamos o potencial evocado (ERP) no eletrodo parietal Pz para o primeiro participante e a matriz de confusao agregada normalizada.



In [ ]:
# Plota a comparação das formas de onda medias no eletrodo Pz
mne.viz.plot_compare_evokeds(
    {name: first_epochs[name].average() for name in ["standard", "target"]},
    picks="Pz",
    show=False,
)
# Plota a matriz de confusao agregada normalizada pelas classes verdadeiras
ConfusionMatrixDisplay.from_predictions(
    y, predictions, display_labels=["standard", "target"], normalize="true"
)
plt.show()

## Conclusao
Extraimos caracteristicas de amplitude temporal do componente P300 e avaliamos sua capacidade de generalização entre sujeitos usando validação cruzada sem vazamento de dados.

